In [1]:
import numpy as np
from analyze import compute_training_states, get_sorted_param_paths, compute_transitions, compute_test_states
from tqdm.auto import tqdm


# checkpoints
folder = "../exp/checkpoints/"
param_paths = get_sorted_param_paths(folder)

/raven/u/nmilosevic/workspace/ogbench/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.environ["MUJOCO_GL"]="osmesa"

In [3]:
from analyze import compute_training_dirichlet_energy
# Compute the training states for 100 trajectories, for the first 100 training steps (param_paths[:100]) and the fully trained model
train_states = compute_training_dirichlet_energy(folder, param_paths[:100] + param_paths[-1:], n_trajectories=100, include_initial=True)

100%|██████████| 4/4 [00:45<00:00, 11.32s/it]


In [4]:
from analyze import compute_test_dirichlet_energy
test_states = compute_test_dirichlet_energy(folder, param_paths[-1], n_trajectories=100, pbar=False)

Restored from ../exp/checkpoints/params_1000000.pkl


100%|██████████| 100/100 [34:23<00:00, 20.64s/it]


In [5]:
final_train_energies = np.concatenate(train_states[list(train_states.keys())[-1]][100:]).mean(-1)

In [6]:
# compute success / fail
densities_success, densities_fail = [], []
for trajectory in tqdm(test_states):
    states = trajectory[0]
    if trajectory[1]:  # If the trajectory was successful
        densities_success.append(np.array(states).mean())
    else:  # If the trajectory failed
        densities_fail.append(np.array(states).mean())
    

100%|██████████| 100/100 [00:00<00:00, 7736.43it/s]


In [7]:
final_test_success_energies = np.array(densities_success)

In [8]:
final_test_fail_energies = np.array(densities_fail)

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

sns.boxplot(y=final_train_energies, x=0, label=f"Train Trajectories ({len(final_train_energies)})")
sns.boxplot(y=np.concatenate([final_test_success_energies, final_test_fail_energies]), x=1, label=f"Test Trajectories ({len(final_test_success_energies) + len(final_test_fail_energies)})")
sns.boxplot(y=final_test_success_energies, x=2, label=f"Successful Test Trajectories ({len(final_test_success_energies)})")
sns.boxplot(y=final_test_fail_energies, x=3, label=f"Failed Test Trajectories ({len(final_test_fail_energies)})")
plt.ylabel("Dirichlet Energy")
plt.gca().get_xaxis().set_visible(False)
#plt.ylim(0.96, 1.01)
plt.savefig("geometric_complexity.png", dpi=300)